# SignBridge - word model training on a Kaggle GPU

1. **Add Data** -> search *Indian Sign Language Mediapipe Holistic Landmarks* (swaptr) and add it.
2. **Settings -> Accelerator -> GPU**, then **Run All**.
3. Download `word_model.onnx`, `word_model.pt`, `labels.json` (-> `models/`) and `metrics.json`, `*.png` (-> `experiments/kaggle/`) from the Output tab.

The same code also runs on a laptop CPU in a few minutes (`python -m ml.train`), because the model is small.

In [ ]:
import glob, json, os, random, time
import numpy as np, pandas as pd, torch
from torch import nn
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
INPUT = glob.glob('/kaggle/input/*/keypoints')[0] if glob.glob('/kaggle/input/*/keypoints') else 'data/raw/include/keypoints'
OUT = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
EPOCHS, SEED = 150, 42
print(DEVICE, INPUT)

## Vocabulary and landmark features (same code as `ml/vocab.py`, `ml/features.py`)

In [ ]:
"""Starting vocabulary: 50 everyday ISL words + fingerspelling A-Z and 1-9."""

# gloss -> INCLUDE folder (category/word). "text" is how the gloss reads in a sentence.
_WORDS = [
    # greetings
    ("HELLO", "greetings/hello", "hello"),
    ("HOW_ARE_YOU", "greetings/howareyou", "how are you"),
    ("THANK_YOU", "greetings/thankyou", "thank you"),
    ("GOOD_MORNING", "greetings/goodmorning", "good morning"),
    ("GOOD_NIGHT", "greetings/goodnight", "good night"),
    ("ALRIGHT", "greetings/alright", "alright"),
    ("PLEASED", "greetings/pleased", "pleased"),
    # pronouns
    ("I", "pronouns/i", "I"),
    ("YOU", "pronouns/you", "you"),
    ("HE", "pronouns/he", "he"),
    ("SHE", "pronouns/she", "she"),
    ("WE", "pronouns/we", "we"),
    ("THEY", "pronouns/they", "they"),
    # people
    ("MOTHER", "people/mother", "mother"),
    ("FATHER", "people/father", "father"),
    ("BROTHER", "people/brother", "brother"),
    ("SISTER", "people/sister", "sister"),
    ("FRIEND", "people/friend", "friend"),
    ("FAMILY", "people/family", "family"),
    ("BABY", "people/baby", "baby"),
    ("DOCTOR", "jobs/doctor", "doctor"),
    ("TEACHER", "jobs/teacher", "teacher"),
    ("POLICE", "jobs/police", "police"),
    # places
    ("HOSPITAL", "places/hospital", "hospital"),
    ("HOUSE", "places/house", "house"),
    ("SCHOOL", "places/school", "school"),
    ("MARKET", "places/market", "market"),
    ("BANK", "places/bank", "bank"),
    ("OFFICE", "places/office", "office"),
    ("RESTAURANT", "places/restaurant", "restaurant"),
    # describing words
    ("GOOD", "adjectives/good", "good"),
    ("BAD", "adjectives/bad", "bad"),
    ("HAPPY", "adjectives/happy", "happy"),
    ("SAD", "adjectives/sad", "sad"),
    ("SICK", "adjectives/sick", "sick"),
    ("HOT", "adjectives/hot", "hot"),
    ("COLD", "adjectives/cold", "cold"),
    ("NEW", "adjectives/new", "new"),
    ("FAST", "adjectives/fast", "fast"),
    ("SLOW", "adjectives/slow", "slow"),
    # time
    ("TODAY", "days_and_time/today", "today"),
    ("TOMORROW", "days_and_time/tomorrow", "tomorrow"),
    ("YESTERDAY", "days_and_time/yesterday", "yesterday"),
    ("MORNING", "days_and_time/morning", "morning"),
    ("NIGHT", "days_and_time/night", "night"),
    ("TIME", "days_and_time/time", "time"),
    # things
    ("MEDICINE", "society/medicine", "medicine"),
    ("MONEY", "society/money", "money"),
    ("BOOK", "home/book", "book"),
    ("BATHROOM", "home/bathroom", "bathroom"),
]

WORDS = [{"gloss": g, "source": s, "text": t} for g, s, t in _WORDS]
WORD_GLOSSES = [w["gloss"] for w in WORDS]
WORD_TEXT = {w["gloss"]: w["text"] for w in WORDS}

LETTERS = [chr(c) for c in range(ord("A"), ord("Z") + 1)] + [str(d) for d in range(1, 10)]

# Folder names in the "ISL words with landmarks" video dataset that match our glosses.
WORD_VIDEO_FOLDERS = {w["source"].split("/")[1]: w["gloss"] for w in WORDS}

In [ ]:
"""Landmark -> feature conversion shared by training, evaluation and the live backend.

Words: every frame keeps 7 upper-body pose points + 21 left-hand + 21 right-hand points
(x, y only), in body coordinates (origin = shoulder centre, unit = shoulder width).
A sign is a sequence of such frames, trimmed to the hands-visible part and resampled to T frames.

Letters: one frame, up to two hands, described by their shape (relative to the wrist) and
the offset between the two hands.
"""
import numpy as np

T = 32  # frames per word sample
POSE_IDX = [0, 11, 12, 13, 14, 15, 16]  # nose, shoulders, elbows, wrists
POSE_MIRROR = [0, 2, 1, 4, 3, 6, 5]  # left/right swap inside POSE_IDX
N_POINTS = len(POSE_IDX) + 42  # 49
WORD_DIM = 14 + 42 + 42 + 42 + 42 + 2  # 184
LETTER_DIM = 42 + 42 + 3  # 87


# ---------------------------------------------------------------- words
def body_frame(pose, left, right, aspect):
    """pose (33,2+) / left (21,2+) / right (21,2+) in normalised image coords, NaN or None when missing.
    Returns (49, 2) float32 in body coordinates (NaN where missing)."""
    pts = np.full((N_POINTS, 2), np.nan, np.float32)
    if pose is not None:
        pts[: len(POSE_IDX)] = np.asarray(pose, np.float32)[POSE_IDX, :2]
    if left is not None:
        pts[7:28] = np.asarray(left, np.float32)[:, :2]
    if right is not None:
        pts[28:49] = np.asarray(right, np.float32)[:, :2]
    pts[:, 0] *= aspect  # make x and y the same unit
    ls, rs = pts[1], pts[2]
    if np.isfinite(ls).all() and np.isfinite(rs).all() and np.linalg.norm(ls - rs) > 1e-3:
        centre, scale = (ls + rs) / 2, np.linalg.norm(ls - rs)
    else:  # no body found: assume a centred signer
        centre, scale = np.array([0.5 * aspect, 0.55], np.float32), 0.35
    return (pts - centre) / scale


def has_hand(frame):
    return np.isfinite(frame[7:49, 0]).any()


def resample_sequence(frames, t=T, margin=2):
    """frames (n, 49, 2) -> (t, 49, 2): keep the hands-visible span, pick t evenly spaced frames."""
    frames = np.asarray(frames, np.float32)
    active = np.where([has_hand(f) for f in frames])[0]
    if len(active):
        frames = frames[max(active[0] - margin, 0): active[-1] + margin + 1]
    idx = np.linspace(0, len(frames) - 1, t).round().astype(int)
    return frames[idx]


def _hand_local(h):
    """(21,2) body coords -> shape relative to wrist, scaled by wrist->middle-knuckle length."""
    size = np.linalg.norm(h[9] - h[0]) + 1e-6
    return (h - h[0]) / size


def word_features(seq, mirror=False):
    """seq (T, 49, 2) body coords -> (T, 184) float32 model input."""
    seq = np.array(seq, np.float32)
    if mirror:
        seq[..., 0] *= -1
        pose = seq[:, :7][:, POSE_MIRROR]
        seq = np.concatenate([pose, seq[:, 28:49], seq[:, 7:28]], axis=1)
    out = np.zeros((len(seq), WORD_DIM), np.float32)
    for i, f in enumerate(seq):
        pose, lh, rh = f[:7], f[7:28], f[28:49]
        out[i, :14] = np.nan_to_num(pose).ravel()
        for k, h in enumerate((lh, rh)):
            if np.isfinite(h).all():
                out[i, 14 + 42 * k: 56 + 42 * k] = h.ravel()
                out[i, 98 + 42 * k: 140 + 42 * k] = _hand_local(h).ravel()
                out[i, 182 + k] = 1.0
    return out


# ---------------------------------------------------------------- letters
def letter_features(hands, mirror=False):
    """hands: list of (21,2) arrays in aspect-corrected image coords (x already * aspect).
    Returns (87,) float32. Hands are ordered left-to-right in the image."""
    hands = [np.array(h, np.float32)[:, :2] for h in hands][:2]
    if mirror:
        for h in hands:
            h[:, 0] *= -1
    hands.sort(key=lambda h: h[0, 0])
    out = np.zeros(LETTER_DIM, np.float32)
    if not hands:
        return out
    out[:42] = _hand_local(hands[0]).ravel()
    if len(hands) > 1:
        out[42:84] = _hand_local(hands[1]).ravel()
        size = np.linalg.norm(hands[0][9] - hands[0][0]) + 1e-6
        out[84:86] = (hands[1][0] - hands[0][0]) / size
        out[86] = 1.0
    return out

In [ ]:
"""Small PyTorch models: a landmark-sequence Transformer for words, an MLP for letters."""
import torch
from torch import nn




class WordTransformer(nn.Module):
    def __init__(self, n_classes, d=128, layers=2, heads=4, dropout=0.2):
        super().__init__()
        self.embed = nn.Sequential(nn.Linear(WORD_DIM, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout))
        self.pos = nn.Parameter(torch.zeros(1, T, d))
        layer = nn.TransformerEncoderLayer(d, heads, d * 2, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(dropout), nn.Linear(d, n_classes))

    def forward(self, x):  # (B, T, WORD_DIM) -> logits
        h = self.encoder(self.embed(x) + self.pos)
        return self.head(h.mean(dim=1))


class LetterMLP(nn.Module):
    def __init__(self, n_classes, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LETTER_DIM, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)

## Load the 50-word subset and build 32-frame sequences

In [ ]:
INCLUDE_ASPECT = 16 / 9

def points(g, n):
    arr = np.full((n, 2), np.nan, np.float32)
    arr[g.landmark_index.values] = g[['x', 'y']].values
    return None if np.isnan(arr).all() else arr

def take(path):
    df = pd.read_parquet(path, columns=['frame', 'type', 'landmark_index', 'x', 'y'])
    df = df[df.type != 'face']
    frames = []
    for _, f in df.groupby('frame', sort=True):
        p = {t: g for t, g in f.groupby('type')}
        frames.append(body_frame(points(p['pose'], 33) if 'pose' in p else None,
                                 points(p['left_hand'], 21) if 'left_hand' in p else None,
                                 points(p['right_hand'], 21) if 'right_hand' in p else None, INCLUDE_ASPECT))
    return resample_sequence(frames)

rng = random.Random(SEED)
train, test = [], []
for i, w in enumerate(WORDS):
    files = sorted(glob.glob(f"{INPUT}/{w['source']}/*.parquet"))
    rng.shuffle(files)
    n_test = max(2, round(len(files) * 0.2))
    for k, f in enumerate(files):
        (test if k < n_test else train).append((take(f), i))
print(len(train), 'train', len(test), 'test')

## Train (augmentation: rotation, scale, shift, time warp, mirrored signing)

In [ ]:
def augment(seq, r):
    ang = np.deg2rad(r.uniform(-12, 12))
    rot = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]], np.float32)
    seq = seq @ rot.T * r.uniform(0.85, 1.15) + r.uniform(-0.12, 0.12, 2).astype(np.float32)
    idx = np.sort(np.clip(np.arange(T) + r.normal(0, 1.2, T), 0, T - 1)).round().astype(int)
    return seq[idx] + r.normal(0, 0.01, seq.shape).astype(np.float32)

nr = np.random.default_rng(SEED)
Xte = torch.tensor(np.stack([word_features(s) for s, _ in test])).to(DEVICE)
yte = torch.tensor([y for _, y in test]).to(DEVICE)
torch.manual_seed(SEED)
model = WordTransformer(len(WORDS)).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=EPOCHS)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
hist, best, best_state, t0 = [], 0, None, time.time()
for ep in range(EPOCHS):
    model.train()
    X = torch.tensor(np.stack([word_features(augment(s, nr), mirror=nr.random() < 0.5) for s, _ in train])).to(DEVICE)
    Y = torch.tensor([y for _, y in train]).to(DEVICE)
    perm = torch.randperm(len(Y))
    for i in range(0, len(perm), 64):
        b = perm[i:i + 64]
        opt.zero_grad(); loss = loss_fn(model(X[b]), Y[b]); loss.backward(); opt.step()
    sched.step()
    model.eval()
    with torch.no_grad():
        acc = (model(Xte).argmax(1) == yte).float().mean().item()
    hist.append({'epoch': ep + 1, 'loss': loss.item(), 'test_acc': acc})
    if acc >= best:
        best, best_state = acc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if (ep + 1) % 10 == 0:
        print(ep + 1, round(loss.item(), 3), round(acc, 3))
print('best test accuracy', best, 'in', round(time.time() - t0), 's')

## Export ONNX, metrics and plots

In [ ]:
import matplotlib.pyplot as plt
model = WordTransformer(len(WORDS)); model.load_state_dict(best_state); model.eval()
torch.save(best_state, f'{OUT}/word_model.pt')
torch.onnx.export(model, torch.zeros(1, T, WORD_DIM), f'{OUT}/word_model.onnx', input_names=['x'], output_names=['logits'],
                  dynamic_axes={'x': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17, dynamo=False)
with torch.no_grad():
    pred = model(Xte.cpu()).argmax(1).numpy()
cm = np.zeros((len(WORDS), len(WORDS)), int)
for a, b in zip(yte.cpu().numpy(), pred):
    cm[a, b] += 1
json.dump({'words': WORD_GLOSSES, 'letters': LETTERS}, open(f'{OUT}/labels.json', 'w'), indent=2)
json.dump({'test_accuracy': best, 'epochs': EPOCHS, 'device': DEVICE, 'train_samples': len(train),
           'test_samples': len(test), 'confusion': cm.tolist(), 'history': hist}, open(f'{OUT}/metrics.json', 'w'), indent=1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot([h['test_acc'] for h in hist]); ax[0].set_title('test accuracy')
ax[1].imshow(cm, cmap='Blues'); ax[1].set_title('confusion matrix')
fig.savefig(f'{OUT}/training.png', dpi=110)
print(os.listdir(OUT))